In [0]:
# Test if your agent endpoint exists
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List your serving endpoints
print("Available endpoints:")
for endpoint in w.serving_endpoints.list():
    print(f"  - {endpoint.name} | State: {endpoint.state.config_update}")

# Check if your GDPR agent endpoint exists
try:
    endpoint = w.serving_endpoints.get(name="gdpr-agent-endpoint")
    print(f"\n✅ Found endpoint: {endpoint.name}")
    print(f"   State: {endpoint.state.config_update}")
except Exception as e:
    print(f"\n❌ Endpoint not found: {e}")
    print("\n⚠️  You need to deploy your GDPR agent first!")

In [0]:
# Test a single query
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

test_question = "What are the rules for data retention under GDPR?"

try:
    response = w.serving_endpoints.query(
        name="gdpr-agent-endpoint",  # Replace with your actual endpoint
        messages=[
            ChatMessage(role=ChatMessageRole.USER, content=test_question)
        ]
    )
    
    print("✅ Agent responded successfully!")
    print(f"Response: {response}")
    
except Exception as e:
    print(f"❌ Agent query failed: {e}")
    print("\n⚠️  Fix this before building evaluation pipeline!")

In [0]:
import json

# Load golden set
with open("/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/evaluation_data/golden_set_v3_production_30.json", 'r') as f:
    dataset = json.load(f)

# Review first 5 cases
for case in dataset["test_cases"][:5]:
    print(f"\n{'='*80}")
    print(f"Case ID: {case['id']}")
    print(f"Category: {case['category']}")
    print(f"Question: {case['question']}")
    print(f"Expected: {case['expected_behavior']}")
    print(f"Evaluation: {case['evaluation_method']}")
    
# Ask yourself:
# - Are these questions realistic?
# - Are the expectations achievable?
# - Do the evaluation methods make sense?

In [0]:
# Manual test of evaluation harness
import sys
sys.path.insert(0, '/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/eval_harness')

from agent_interface.gdpr_agent_client import GDPRAgentClient
from evaluators.base_evaluator import CompositeEvaluator
from evaluators.retrieval_evaluator import RetrievalEvaluator
from evaluators.content_evaluator import ContentEvaluator

# Initialize
agent_client = GDPRAgentClient("gdpr-agent-endpoint")
evaluator = CompositeEvaluator([
    RetrievalEvaluator(),
    ContentEvaluator()
])

# Test on first 3 cases
for test_case in dataset["test_cases"][:3]:
    print(f"\n{'='*80}")
    print(f"Testing: {test_case['id']}")
    
    try:
        # Query agent
        response = agent_client.query(test_case["question"])
        
        # Evaluate
        result = evaluator.evaluate(test_case, response)
        
        print(f"Score: {result['score']:.2f}")
        print(f"Passed: {result['passed']}")
        print(f"Feedback: {result['feedback']}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

# Ask yourself:
# - Does the scoring make sense?
# - Are the thresholds (70%, 80%) realistic?
# - Do you need to adjust evaluation logic?

In [0]:
# Run full evaluation to establish baseline
from eval_harness.evaluate import main
import os

# Set config
os.environ['AGENT_ENDPOINT'] = 'gdpr-agent-endpoint'
os.environ['MLFLOW_EXPERIMENT_NAME'] = '/Users/vernonc.lam@gmail.com/gdpr-agent-evaluation'

# Run
try:
    exit_code = main()
    print(f"\nBaseline evaluation complete. Exit code: {exit_code}")
except Exception as e:
    print(f"❌ Evaluation failed: {e}")
    print("\n⚠️  Fix issues before setting up automation!")